In [1]:
import pandas as pd
df = pd.read_csv("dirty_customer_dataset.csv")
df.head()

,CustomerID,Full Name,Email,Age,Gender,City,Category,Quantity,Price,PaymentMethod,PurchaseDate
0,CUST09542,Eric Lowery,brownamy@example.org,53.0,Male,CHENNAI,Furniture,5,374.41,Net Banking,"April 25, 2026"
1,CUST07535,Jeremy Davis,andrew47@example.net,42.0,F,CHENNAI,Beauty,6,2680.77,cash,09/16/2024
2,CUST04697,Cody Davidson,jamesclarke@example.org,62.0,Male,CHENNAI,Books,2,219.56,COD,20 Apr 2026
3,CUST02993,Benjamin Sullivan,gilllaura@example.org,NaN,Female,Calcutta,Grocery,6,744.22,UPI,28/07/2026
4,CUST05503,Kristen Ramirez,glenmiller@example.org,44.0,FEMALE,Calcutta,Toys,10,1713.84,DEBIT CARD,19/12/2024


## Data Quality Report


In [2]:
# 1. Basic shape and info
print("Shape:", df.shape)
df.info()

Shape: (11056, 11)
<class 'pandas.DataFrame'>
RangeIndex: 11056 entries, 0 to 11055
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CustomerID     11025 non-null  str    
 1   Full Name      11025 non-null  str    
 2   Email          10911 non-null  str    
 3   Age            10653 non-null  float64
 4   Gender         10007 non-null  str    
 5   City           11025 non-null  str    
 6   Category       10957 non-null  str    
 7   Quantity       10928 non-null  str    
 8   Price          10785 non-null  str    
 9   PaymentMethod  10899 non-null  str    
 10  PurchaseDate   10709 non-null  str    
dtypes: float64(1), str(10)
memory usage: 950.3 KB


In [3]:
# 2. Null counts per column
print("Missing values per column:")
print(df.isna().sum())

Missing values per column:
CustomerID         31
Full Name          31
Email             145
Age               403
Gender           1049
City               31
Category           99
Quantity          128
Price             271
PaymentMethod     157
PurchaseDate      347
dtype: int64


In [4]:
# 3. Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 555


In [5]:
# 4. Data type issues (columns that should be numeric but are stored as text)
print(df.dtypes)
print(df[["Age", "Quantity", "Price"]].head(15))

CustomerID           str
Full Name            str
Email                str
Age              float64
Gender               str
City                 str
Category             str
Quantity             str
Price                str
PaymentMethod        str
PurchaseDate         str
dtype: object
     Age Quantity     Price
0   53.0        5    374.41
1   42.0        6   2680.77
2   62.0        2    219.56
3    NaN        6    744.22
4   44.0       10   1713.84
5   34.0        7   1277.13
6   28.0        8    636.63
7   46.0        7  119521.0
8   40.0        6   1189.99
9   30.0        5   1486.84
10  47.0        1    346.32
11  18.0        4   2373.57
12  20.0        8   1925.86
13  31.0       10    172.01
14  25.0        5    813.87


In [6]:
# 5. Value range anomalies (quick look at min/max on numeric-looking columns)
# Note: Age, Quantity, Price may still be object dtype due to mixed values - we'll check raw ranges first
print(df["Age"].describe())

count    10653.000000
mean        41.198442
std         56.313565
min         -5.000000
25%         28.000000
50%         38.000000
75%         47.000000
max        999.000000
Name: Age, dtype: float64


## Missing Data Handling

In [8]:
import numpy as np
# Step 1: Treat invalid ages (negative, 0, or unrealistically high) as missing too
df.loc[(df["Age"] < 18) | (df["Age"] > 100), "Age"] = np.nan

# Step 2: Now impute missing Age with median (robust to outliers, better than mean here)
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)

print("Missing Age after cleaning:", df["Age"].isna().sum())
print(df["Age"].describe())

Missing Age after cleaning: 0
count    11056.000000
mean        37.945459
std         12.579049
min         18.000000
25%         29.000000
50%         38.000000
75%         46.000000
max         75.000000
Name: Age, dtype: float64


**Justification:** Age values like -5, 0, and 999 are clearly invalid data entry errors, not real ages — 
treated as missing. Median imputation is used instead of mean because Age had extreme outliers before cleaning, 
which would have skewed the mean upward.

### Gender

In [9]:
# Check current messy values first
print(df["Gender"].unique())

<StringArray>
[  '  Male',        'F',     'Male',   'Female',   'FEMALE',   'female',
 'Female  ',     'male',     'MALE',        'M',        nan]
Length: 11, dtype: str


In [10]:
# Step 1: Standardize inconsistent formatting
df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()
df["Gender"] = df["Gender"].replace({
    "MALE": "Male", "M": "Male",
    "FEMALE": "Female", "F": "Female",
    "NAN": np.nan  # restore true NaN (since .astype(str) turns NaN into the string "nan")
})

print(df["Gender"].unique())
print("Missing Gender:", df["Gender"].isna().sum())

<StringArray>
['Male', 'Female', nan]
Length: 3, dtype: str
Missing Gender: 1049


In [11]:
# Step 2: Fill missing Gender with mode (most frequent value)
mode_gender = df["Gender"].mode()[0]
df["Gender"] = df["Gender"].fillna(mode_gender)

print("Missing Gender after cleaning:", df["Gender"].isna().sum())
print(df["Gender"].value_counts())

Missing Gender after cleaning: 0
Gender
Female    6111
Male      4945
Name: count, dtype: int64


**Justification:** Gender had many inconsistent formats (Male/male/MALE/M/  Male ) which were first standardized. 
Missing values were filled with the mode (most common value) since Gender is categorical and mode imputation is 
the standard approach — better than dropping ~1,000 rows (10% of the dataset).

### City

In [12]:
# Check current messy values first
print(df["City"].unique())

<StringArray>
[  'CHENNAI',  'Calcutta', 'Bangalore',      'pune', 'hyderabad',   'KOLKATA',
   ' mumbai',  ' Chennai', 'bangalore',     'DELHI',      'Pune', 'new delhi',
   'kolkata',   'Kolkata', 'Hyderabad',   'chennai',    'MUMBAI',    'mumbai',
     'Pune ',   'Chennai',    'Mumbai', 'bengaluru',     'delhi', 'BANGALORE',
 'Bengaluru', 'New Delhi', 'HYDERABAD',      'PUNE',         nan,   'Mumbai ',
     'Delhi']
Length: 31, dtype: str


In [13]:
# Step 1: Standardize inconsistent formatting
df["City"] = df["City"].astype(str).str.strip().str.title()

# Fix known duplicates that title-casing alone won't merge
df["City"] = df["City"].replace({
    "New Delhi": "Delhi",
    "Bengaluru": "Bangalore",
    "Nan": np.nan
})

print(df["City"].unique())
print("Missing City:", df["City"].isna().sum())

<StringArray>
[  'Chennai',  'Calcutta', 'Bangalore',      'Pune', 'Hyderabad',   'Kolkata',
    'Mumbai',     'Delhi',         nan]
Length: 9, dtype: str
Missing City: 31


In [14]:
# Step 2: Fill missing City with mode
mode_city = df["City"].mode()[0]
df["City"] = df["City"].fillna(mode_city)

print("Missing City after cleaning:", df["City"].isna().sum())
print(df["City"].value_counts())

Missing City after cleaning: 0
City
Chennai      1658
Pune         1608
Bangalore    1579
Hyderabad    1579
Delhi        1548
Mumbai       1536
Kolkata      1146
Calcutta      402
Name: count, dtype: int64


**Justification:** City had inconsistent casing and spacing (mumbai/MUMBAI/ mumbai), plus two naming variants 
(New Delhi→Delhi, Bengaluru→Bangalore) that were merged for consistency. Missing values (only 31, <1%) were 
filled with the mode since City is categorical with a clear most-common value.

### Category

In [15]:
# Check current messy values first
print(df["Category"].unique())

<StringArray>
[     'Furniture',         'Beauty',          'Books',        'Grocery',
           'Toys', 'Home & Kitchen',    'Electronics',       'Footwear',
         'Sports',       'Clothing',              nan,       'footwear',
         'beauty', 'home & kitchen',         'sports',      'furniture',
       'clothing',        'grocery',          'books',    'electronics',
           'toys']
Length: 21, dtype: str


In [16]:
# Step 1: Standardize casing
df["Category"] = df["Category"].astype(str).str.strip().str.title()
df["Category"] = df["Category"].replace({"Nan": np.nan})

print(df["Category"].unique())
print("Missing Category:", df["Category"].isna().sum())

<StringArray>
[     'Furniture',         'Beauty',          'Books',        'Grocery',
           'Toys', 'Home & Kitchen',    'Electronics',       'Footwear',
         'Sports',       'Clothing',              nan]
Length: 11, dtype: str
Missing Category: 99


In [17]:
# Step 2: Fill missing Category with a placeholder (not mode, since category is a product choice, 
# not something we should guess/assume for a customer)
df["Category"] = df["Category"].fillna("Unknown")

print("Missing Category after cleaning:", df["Category"].isna().sum())
print(df["Category"].value_counts())

Missing Category after cleaning: 0
Category
Home & Kitchen    1164
Sports            1138
Books             1108
Electronics       1107
Clothing          1095
Footwear          1084
Grocery           1078
Furniture         1076
Beauty            1066
Toys              1041
Unknown             99
Name: count, dtype: int64


**Justification:** Category had minor casing inconsistencies (electronics vs Electronics) which were standardized. 
For missing values (99 rows, <1%), we used "Unknown" instead of mode — since guessing a specific product category 
a customer purchased could introduce false analysis (e.g. inflating a category's sales count incorrectly).

### Quantity

In [18]:
# Check current messy values first
print(df["Quantity"].unique()[:30])

<StringArray>
['5', '6', '2', '10', '7', '8', '1', '4', '3', '9', '-1', '0', 'five', nan,
 '-3']
Length: 15, dtype: str


In [19]:
# Step 1: Convert to numeric, forcing invalid text (like "five") to NaN
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

print("Missing Quantity after numeric conversion:", df["Quantity"].isna().sum())
print(df["Quantity"].describe())

Missing Quantity after numeric conversion: 236
count    10820.000000
mean         5.384658
std          2.923416
min         -3.000000
25%          3.000000
50%          5.000000
75%          8.000000
max         10.000000
Name: Quantity, dtype: float64


In [20]:
# Step 2: Treat invalid values (negative or zero quantity) as missing too
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan

print("Missing Quantity after removing invalid values:", df["Quantity"].isna().sum())

Missing Quantity after removing invalid values: 338


In [21]:
# Step 3: Fill missing Quantity with median
median_qty = df["Quantity"].median()
df["Quantity"] = df["Quantity"].fillna(median_qty)

print("Missing Quantity after cleaning:", df["Quantity"].isna().sum())
print(df["Quantity"].describe())

Missing Quantity after cleaning: 0
count    11056.000000
mean         5.433792
std          2.820092
min          1.000000
25%          3.000000
50%          5.000000
75%          8.000000
max         10.000000
Name: Quantity, dtype: float64


**Justification:** Quantity had text entries ("five") that couldn't convert to numbers, plus invalid values 
like 0 or negative quantities (can't purchase negative items) — both treated as missing. Median imputation was 
used since Quantity is a discrete count with likely outliers, making median more robust than mean.

### Price

In [22]:
# Check current messy values first
print(df["Price"].unique()[:30])

<StringArray>
[  '374.41',  '2680.77',   '219.56',   '744.22',  '1713.84',  '1277.13',
   '636.63', '119521.0',  '1189.99',  '1486.84',   '346.32',  '2373.57',
  '1925.86',   '172.01',   '813.87',  '4059.36',  '2312.03',   '857.34',
  '3473.65',   '1996.0',        nan,    '150.8',   '4092.2',  '1736.13',
   '694.96',   '782.66',   '347.05',  '1099.44',  '1314.94',   '1254.6']
Length: 30, dtype: str


In [23]:
# Step 1: Remove currency symbols and convert to numeric
df["Price"] = df["Price"].astype(str).str.replace("$", "", regex=False)
df["Price"] = df["Price"].str.replace("₹", "", regex=False)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

print("Missing Price after numeric conversion:", df["Price"].isna().sum())
print(df["Price"].describe())

Missing Price after numeric conversion: 271
count    1.078500e+04
mean     3.262003e+03
std      2.460281e+04
min     -9.426330e+03
25%      4.718500e+02
50%      1.099150e+03
75%      2.169590e+03
max      1.007696e+06
Name: Price, dtype: float64


In [24]:
# Step 2: Treat negative prices as missing (invalid - price can't be negative)
df.loc[df["Price"] < 0, "Price"] = np.nan

print("Missing Price after removing negatives:", df["Price"].isna().sum())

Missing Price after removing negatives: 369


In [25]:
# Step 3: Detect extreme outliers using IQR method (before filling missing values)
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("IQR bounds:", lower_bound, "to", upper_bound)
outliers = df[(df["Price"] < lower_bound) | (df["Price"] > upper_bound)]
print("Number of outliers:", len(outliers))

IQR bounds: -2061.3899999999994 to 4731.889999999999
Number of outliers: 592


In [26]:
# Step 4: Cap outliers instead of removing (preserves row count, avoids losing other column data)
df["Price"] = df["Price"].clip(lower=lower_bound, upper=upper_bound)

print(df["Price"].describe())

count    10687.000000
mean      1517.536432
std       1309.044141
min         50.070000
25%        486.090000
50%       1111.840000
75%       2184.410000
max       4731.890000
Name: Price, dtype: float64


In [27]:
# Step 5: Fill remaining missing Price with median
median_price = df["Price"].median()
df["Price"] = df["Price"].fillna(median_price)

print("Missing Price after cleaning:", df["Price"].isna().sum())

Missing Price after cleaning: 0


**Justification:** Price had currency symbols ($, ₹) preventing numeric conversion, negative values (invalid), 
and extreme outliers (100x normal price, likely data entry errors). Outliers were **capped** (not removed) using 
the IQR method to preserve row count while limiting their distorting effect on analysis. Remaining missing values 
were filled with median, robust to the outliers we just capped.

### PaymentMethod

In [28]:
print(df["PaymentMethod"].unique())

<StringArray>
[     'Net Banking',             'cash',              'COD',
              'UPI',       'DEBIT CARD',       'debit card',
       'NETBANKING',      'Credit_Card',      'CREDIT CARD',
      'net banking',       'Debit_Card',      'Credit Card',
             'Cash',             'CASH',       'Debit Card',
               'DC',       'Netbanking',              'Upi',
               'CC',              'upi', 'Cash on Delivery',
                nan,      'credit card']
Length: 23, dtype: str


In [29]:
df["PaymentMethod"] = df["PaymentMethod"].astype(str).str.strip().str.title()
df["PaymentMethod"] = df["PaymentMethod"].replace({
    "Cc": "Credit Card", "Credit_Card": "Credit Card",
    "Dc": "Debit Card", "Debit_Card": "Debit Card",
    "Cod": "Cash", "Cash On Delivery": "Cash",
    "Upi": "UPI", "Netbanking": "Net Banking",
    "Nan": np.nan
})
mode_payment = df["PaymentMethod"].mode()[0]
df["PaymentMethod"] = df["PaymentMethod"].fillna(mode_payment)
print(df["PaymentMethod"].value_counts())

PaymentMethod
Credit Card    2401
Cash           2200
Debit Card     2180
Net Banking    2139
UPI            2136
Name: count, dtype: int64


**Justification:** PaymentMethod had many abbreviations and casing variants (CC, credit card, Credit_Card) 
merged into consistent labels. Missing values filled with mode since it's categorical.

### PurchaseDate

In [30]:
df["PurchaseDate"] = df["PurchaseDate"].replace("0000-00-00", np.nan)
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"], format="mixed", errors="coerce")
print("Missing PurchaseDate:", df["PurchaseDate"].isna().sum())
print(df["PurchaseDate"].dtype)

Missing PurchaseDate: 651
datetime64[us]


**Justification:** PurchaseDate mixed multiple formats (YYYY-MM-DD, DD/MM/YYYY, "April 25, 2026") plus invalid 
placeholder dates ("0000-00-00") and impossible dates ("32/13/2023") — all converted using format='mixed', 
with unparseable values becoming NaT. Unlike other columns, we leave these as missing (not imputed) since 
guessing a purchase date would fabricate data — we'll document how many remain missing instead.

### CustomerID, Full Name, Email

In [31]:
# Clean whitespace/casing on text fields
df["Full Name"] = df["Full Name"].astype(str).str.strip().str.title()
df["Full Name"] = df["Full Name"].replace("Nan", np.nan)

# For rows missing CustomerID, Full Name, or Email - these are identifiers, so we drop those rows
# (can't reasonably impute a fake customer ID or name)
before = len(df)
df = df.dropna(subset=["CustomerID", "Full Name", "Email"])
after = len(df)
print(f"Rows dropped due to missing identifiers: {before - after}")

Rows dropped due to missing identifiers: 145


**Justification:** CustomerID, Name, and Email are unique identifiers — imputing fake values would corrupt 
data integrity. Rows missing any of these were dropped instead.

## Duplicate Removal

In [32]:
# Check duplicate count before removal
dupes_before = df.duplicated().sum()
print("Duplicate rows found:", dupes_before)

Duplicate rows found: 519


In [33]:
# Remove exact duplicate rows
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))
print("Duplicate rows remaining:", df.duplicated().sum())

Rows after removing duplicates: 10392
Duplicate rows remaining: 0


**Justification:** 519 exact duplicate rows were identified and removed, as they represent repeated entries 
of the same transaction with no additional information — keeping them would inflate counts and skew analysis.

## Data Type Correction

In [34]:
# Ensure correct dtypes across all columns
df["CustomerID"] = df["CustomerID"].astype(str)
df["Age"] = df["Age"].astype(int)
df["Quantity"] = df["Quantity"].astype(int)
df["Price"] = df["Price"].astype(float)
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"])

df.info()

<class 'pandas.DataFrame'>
Index: 10392 entries, 0 to 11055
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   CustomerID     10392 non-null  str           
 1   Full Name      10392 non-null  str           
 2   Email          10392 non-null  str           
 3   Age            10392 non-null  int64         
 4   Gender         10392 non-null  str           
 5   City           10392 non-null  str           
 6   Category       10392 non-null  str           
 7   Quantity       10392 non-null  int64         
 8   Price          10392 non-null  float64       
 9   PaymentMethod  10392 non-null  str           
 10  PurchaseDate   9800 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(2), str(7)
memory usage: 974.2 KB


**Justification:** CustomerID kept as string (IDs shouldn't be treated as numbers for math). Age and Quantity 
converted to integers since they're whole-number counts. Price kept as float for currency precision. 
PurchaseDate confirmed as proper datetime type.

## Outlier Detection (IQR Method)

In [35]:
# Function to detect outliers using IQR for any numeric column
def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    print(f"{column}: {len(outliers)} outliers found (bounds: {lower:.2f} to {upper:.2f})")
    return lower, upper

for col in ["Age", "Quantity", "Price"]:
    detect_outliers_iqr(col)

Age: 42 outliers found (bounds: 1.00 to 73.00)
Quantity: 0 outliers found (bounds: -4.50 to 15.50)
Price: 614 outliers found (bounds: -1907.70 to 4535.31)


**Findings and decisions:**
- **Age:** Already cleaned earlier (invalid values like -5, 0, 999 treated as missing, then imputed with median) 
  — no further outlier action needed since realistic age range (18-100) was already enforced.
- **Quantity:** Check output above — if outliers exist, they're likely legitimate bulk purchases, so we **retain** 
  them rather than cap/remove, since quantity outliers don't distort analysis as severely as price outliers.
- **Price:** Already capped using IQR bounds earlier (100x-inflated prices were clear data entry errors) — 
  documented and handled in the Price cleaning section above.

## Before vs After Summary

In [ ]:
# Before values (from your original data quality report)
before = {
    "Row Count": 11056,
    "Total Missing Values": 31+31+145+403+1049+31+99+128+271+157+347,
    "Duplicate Rows": 519,
    "Correct Dtypes": "3/11 columns (CustomerID, Full Name, Email as text; rest needed fixing)"
}

# After values (recalculate from current cleaned df)
after = {
    "Row Count": len(df),
    "Total Missing Values": df.isna().sum().sum(),
    "Duplicate Rows": df.duplicated().sum(),
    "Correct Dtypes": "11/11 columns"
}

summary_table = pd.DataFrame([before, after], index=["Before Cleaning", "After Cleaning"])
print(summary_table)

## Save Cleaned Dataset

In [36]:
df.to_csv("customer_data_cleaned.csv", index=False)
print("Cleaned dataset saved successfully.")
print("Final shape:", df.shape)

Cleaned dataset saved successfully.
Final shape: (10392, 11)
